# Causal Forest trên Kaggle — notebook chạy được ngay

Notebook này hoàn tất **hạng mục còn thiếu duy nhất** của dự án: chạy `CausalForestDML`
ở quy mô thật. Laptop 15,19 GB không đủ (ngoại suy 50% cần khoảng 17,5 GB).

## Ba việc bắt buộc trước khi chạy

1. **Settings → Accelerator = `None (CPU)`.** GPU không làm forest nhanh hơn:
   `CausalForestDML` dùng CPU parallelism và system RAM, không dùng CUDA.
2. **Settings → Internet = `On`.** Cần cho `pip install` và `git clone`. Đây là lỗi
   khiến Cell 3 và Cell 4 fail mà thông báo lỗi không nói rõ nguyên nhân.
3. **Add Input → attach dataset Criteo v2.1.** Bấm nút `+` bên phải tên dataset trong
   panel; chỉ tìm thấy dataset trong ô search là **chưa** attach.

## Chạy tuần tự từng cell, không dùng Run All

Cell 5 yêu cầu restart kernel. `Run All` sẽ bỏ qua bước đó và `import econml` ở Cell 6
sẽ lỗi hoặc nạp nhầm phiên bản scikit-learn.

## Về định dạng dữ liệu

Kaggle **giải nén** file `.csv.gz` khi bạn upload trực tiếp. File mount vào
`/kaggle/input` vì thế là CSV thô 3.248.115.221 byte, không phải bản nén 311.422.618
byte. Nội dung hai bản giống nhau bit-for-bit. Cell 2 và gate script chấp nhận cả hai,
mỗi dạng một checksum riêng, nên không cần upload lại dataset.

## Cell 1 — Đọc tài nguyên thật của session

Không giả định Kaggle luôn cấp cùng một cấu hình. Ghi lại `RAM total`: toàn bộ stop rule
tính theo tỷ lệ trên con số này.

Cell này cũng liệt kê mọi file trong `/kaggle/input`. Đọc kỹ phần đó — nó cho biết tên
file thật và dung lượng thật, là hai thứ Kaggle hay đổi lúc upload.

In [ ]:
import os, platform, shutil
import psutil

vm = psutil.virtual_memory()
disk = shutil.disk_usage('/kaggle/working')
print(f'python            {platform.python_version()}')
print(f'logical cpus      {os.cpu_count()}')
print(f'physical cpus     {psutil.cpu_count(logical=False)}')
print(f'RAM total         {vm.total / 2**30:.2f} GB')
print(f'RAM available     {vm.available / 2**30:.2f} GB')
print(f'working disk free {disk.free / 2**30:.2f} GB')
print()

found = False
for root, _, files in os.walk('/kaggle/input'):
    for name in files:
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p):>16,} byte  {p}')
        found = True
if not found:
    print('KHONG CO FILE NAO trong /kaggle/input')
    print('-> Add Input o thanh ben phai, bam nut + ben canh ten dataset.')

if vm.total / 2**30 < 20:
    print()
    print('CANH BAO: RAM total duoi 20 GB. KHONG chay stage 50%.')
    print('Ket thuc o learning curve 20-30% va bao cao dung nhu vay.')

## Cell 2 — Xác minh checksum dữ liệu

Mất khoảng một phút với bản nén, hai đến ba phút với bản CSV thô 3,2 GB. Đáng, vì nó
tránh việc phát hiện sai dataset sau ba tiếng chạy.

Nếu checksum sai thì **đừng sửa hằng số cho khớp**. Assert này tồn tại để bảo đảm bạn
đang chạy đúng Criteo v2.1 nguyên vẹn; kết quả chỉ so được với bảng release Sprint 1 khi
dữ liệu đúng đến từng byte.

In [ ]:
import glob, hashlib, os

# Hai dang byte hop le cua cung mot du lieu:
#   csv.gz    311.422.618 byte  - ban tai ve goc
#   csv     3.248.115.221 byte  - ban Kaggle da giai nen khi upload
SHA_GZ  = '2716e1bf0fd157a93b5bf86924d9088419dfbac2022c6cd90030220634f616dc'
SHA_CSV = 'e4d7c710ca1f38e523309d0f8a0745d1b53e7392d51f20d1088b6cfeaef222ef'

# Pattern rong: Kaggle co the cat 'research-' hoac bo duoi '.gz' khi upload.
matches = sorted(glob.glob('/kaggle/input/**/*criteo*uplift*v2.1.csv*', recursive=True))
assert matches, (
    'Khong tim thay file Criteo. Kiem tra hai thu:\n'
    '  1. Da bam nut + de attach dataset chua (Add Input o thanh ben phai)?\n'
    '  2. Ten file trong dataset co chua "criteo" va "uplift" va "v2.1" khong?\n'
    '     Xem lai output Cell 1 de biet ten that.'
)
DATA_PATH = matches[0]
if len(matches) > 1:
    print('CANH BAO: tim thay nhieu file, dung file dau tien.')
    for m in matches:
        print('   ', m)

expected = SHA_GZ if DATA_PATH.endswith('.gz') else SHA_CSV
digest = hashlib.sha256()
with open(DATA_PATH, 'rb') as handle:
    for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
actual = digest.hexdigest()

print('path  ', DATA_PATH)
print('size  ', f'{os.path.getsize(DATA_PATH):,} byte')
print('dang  ', 'csv.gz' if DATA_PATH.endswith('.gz') else 'csv (Kaggle da giai nen)')
print('sha256', actual)
assert actual == expected, (
    f'Checksum sai.\n  thuc te : {actual}\n  mong doi: {expected}\n'
    'Du lieu khong phai Criteo v2.1 nguyen ven. Tai lai tu '
    'https://ailab.criteo.com/criteo-uplift-prediction-dataset/ va upload lai.'
)
print()
print('checksum OK')

## Cell 3 — Đưa repository vào session

**Cách A** (mặc định) dùng khi repo đã public trên GitHub — đây là trường hợp hiện tại.

**Cách B** dùng khi repo private hoặc Internet = Off: nén repo bỏ `data/`, `.venv/`,
`output/`, `.git/`, upload thành Kaggle Dataset, rồi đặt `USE_GITHUB = False` và điền
`REPO_DATASET` bằng đường dẫn mount thật (xem output Cell 1).

In [ ]:
import os, shutil, subprocess

USE_GITHUB = True
REPO_URL = 'https://github.com/ThanhDatVN/Causal-Uplift-for-Activation-and-Retention.git'
REPO_DATASET = '/kaggle/input/<ten-dataset-repo>'   # chi dung khi USE_GITHUB = False
REPO_DIR = '/kaggle/working/repo'

if not os.path.exists(REPO_DIR):
    if USE_GITHUB:
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    else:
        assert os.path.isdir(REPO_DATASET), f'Khong thay {REPO_DATASET}'
        shutil.copytree(REPO_DATASET, REPO_DIR, dirs_exist_ok=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())
print(sorted(os.listdir('.')))

for required in ('scripts', 'src', 'configs'):
    assert os.path.isdir(required), f'Thieu thu muc {required} trong repo'
for required in ('scripts/kaggle_causal_forest_gate.py', 'scripts/train_causal_forest.py'):
    assert os.path.isfile(required), f'Thieu file {required}'
print('repo OK')

## Cell 4 — Cài dependency có ghim phiên bản

Đây là chỗ hỏng thường gặp nhất. `econml==0.16.0` yêu cầu `scikit-learn>=1.0,<1.7` và
`shap>=0.38.1,<0.49.0`. Image Kaggle thường có scikit-learn mới hơn giới hạn đó.

Cell in ra `exit code`. Khác `0` thì đọc phần stderr ở trên rồi xử lý trước khi đi tiếp;
đừng restart kernel khi cài chưa xong.

In [ ]:
import subprocess, sys

packages = [
    'econml==0.16.0',
    'scikit-learn>=1.4,<1.7',   # rang buoc cung cua econml 0.16
    'shap>=0.38.1,<0.49.0',     # rang buoc cung cua econml 0.16
    'lightgbm>=4.5',
    'psutil',
]
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + packages,
    capture_output=True, text=True,
)
print(result.stdout[-2000:])
print(result.stderr[-2000:])
print('exit code:', result.returncode)
if result.returncode == 0:
    print()
    print('Cai xong. BAY GIO RESTART KERNEL (Run -> Restart session), roi chay Cell 6.')

## Cell 5 — RESTART KERNEL tại đây

> **Bắt buộc.** `Run → Restart session`, rồi chạy tiếp từ Cell 6.
>
> Không restart thì scikit-learn cũ vẫn nằm trong bộ nhớ, và `import econml` sẽ lỗi
> hoặc chạy sai phiên bản. Đây là lỗi số 1 trong danh mục lỗi của runbook.

Sau khi restart, mọi biến Python đều mất — kể cả `DATA_PATH` và `REPO_DIR`. Cell 6 dựng
lại toàn bộ, nên không cần chạy lại Cell 1–4.

## Cell 6 — Sau restart: dựng lại biến và kiểm tra phiên bản

Cell này là cổng kiểm tra: nếu nó chạy hết mà không assert nào nổ thì môi trường đã đúng
và ba stage phía sau chỉ còn là vấn đề thời gian.

In [ ]:
import glob, os

REPO_DIR = '/kaggle/working/repo'
OUTPUT_ROOT = '/kaggle/working/output/causal_forest'
os.chdir(REPO_DIR)

matches = sorted(glob.glob('/kaggle/input/**/*criteo*uplift*v2.1.csv*', recursive=True))
assert matches, 'Khong tim thay du lieu Criteo. Xem lai Cell 1 va Cell 2.'
DATA_PATH = matches[0]

print('cwd      ', os.getcwd())
print('data     ', DATA_PATH)
print('output   ', OUTPUT_ROOT)
print()

import numpy, scipy, sklearn, lightgbm, econml, pandas
for module in (numpy, scipy, sklearn, lightgbm, econml, pandas):
    print(f'{module.__name__:14s} {module.__version__}')

from packaging.version import Version
assert Version(sklearn.__version__) < Version('1.7'), (
    f'scikit-learn {sklearn.__version__} qua moi cho econml 0.16. '
    'Chay lai Cell 4 roi RESTART KERNEL.'
)
assert econml.__version__ == '0.16.0', f'econml {econml.__version__}, can 0.16.0'

import inspect
from econml.dml import CausalForestDML
assert 'inference' in inspect.signature(CausalForestDML.__init__).parameters

print()
print('moi thu OK, san sang chay stage 20%')

## Cell 7 — Hàm chạy một stage

`kaggle_causal_forest_gate.py` tự kiểm sáu điều: checksum dữ liệu, không cho nhảy stage,
exit code của trainer, peak RSS dưới ngưỡng RAM, mọi score hữu hạn, số dòng score khớp
holdout.

Gate **không** chấm chất lượng model — `"quality_not_assessed": true` nằm ngay trong
manifest. Chấm điểm làm ở local bằng `scripts/evaluate_causal_forest.py`.

In [ ]:
import json, subprocess, sys


def stage_slug(frac):
    """Khop ham _slug trong kaggle_causal_forest_gate.py: 0.2 -> '0p2'."""
    return str(frac).replace('.', 'p')


def run_stage(frac):
    """Chay mot stage, in tom tat. Tra manifest neu pass, None neu fail."""
    print(f'=== stage {frac:.0%} bat dau ===', flush=True)
    result = subprocess.run(
        [sys.executable, 'scripts/kaggle_causal_forest_gate.py',
         '--data-path', DATA_PATH,
         '--frac', str(frac),
         '--output-root', OUTPUT_ROOT,
         '--max-ram-fraction', '0.75'],
        capture_output=True, text=True,
    )
    print(result.stdout[-3000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-3000:])
        print(f'stage {frac:.0%} FAIL, exit code {result.returncode}')
        return None

    path = f'{OUTPUT_ROOT}/preflight_{stage_slug(frac)}/gate_manifest.json'
    with open(path) as handle:
        manifest = json.load(handle)
    runtime = manifest['runtime']
    print()
    print('status               ', manifest['status'])
    print('data form            ', manifest['data'].get('form', 'khong ghi'))
    print(f"peak RSS              {runtime['peak_process_tree_rss_gb']:.2f} GB")
    print(f"peak RAM fraction     {runtime['peak_process_tree_ram_fraction']:.3f}")
    print(f"wall time             {runtime['wall_seconds'] / 60:.1f} phut")
    print('may continue         ', manifest['stop_rule']['may_continue'])
    print()
    for nxt in (0.30, 0.50):
        if nxt > frac:
            rss = runtime['peak_process_tree_rss_gb'] * nxt / frac
            minutes = runtime['wall_seconds'] * nxt / frac / 60
            print(f'du bao {nxt:.0%}: RSS ~ {rss:.2f} GB, time ~ {minutes:.0f} phut')
    return manifest

## Cell 8 — Stage 20%

Ước tính 35–60 phút tuỳ CPU của session. Đọc dòng `du bao` ở cuối output: nếu dự báo RSS
cho stage 50% vượt 75% RAM total thì dừng ở 30% và báo cáo learning curve.

In [ ]:
m20 = run_stage(0.20)

## Cell 9 — Stage 30%

Chỉ chạy khi stage 20% `passed` **và** dự báo RSS còn dưới 75% RAM total.

In [ ]:
assert m20 is not None and m20['status'] == 'passed', 'Stage 20% chua pass'
m30 = run_stage(0.30)

## Cell 10 — Stage 50%

Đây là stage **duy nhất** so được trực tiếp với bảng release Sprint 1: ở
`frac=0.50, test_size=0.30, seed=42`, holdout trùng khít final test Sprint 1
(2.096.940 dòng, `Y` và `T` giống hệt từng phần tử — đã kiểm chứng).

Ước tính 2–3,5 giờ. Cộng cả ba stage khoảng 4–5 giờ, nằm trong giới hạn 12 giờ của
session CPU. Nếu thời gian còn lại không đủ, **dừng ở đây** và báo cáo learning curve
20–30%. Đó là kết quả hợp lệ, không phải thất bại.

Trước khi chạy cell này nên bấm **Save Version → Save & Run All (Commit)** nếu muốn chạy
nền; chạy trong cửa sổ interactive sẽ mất hết output nếu mất kết nối.

In [ ]:
assert m30 is not None and m30['status'] == 'passed', 'Stage 30% chua pass'
m50 = run_stage(0.50)

## Cell 11 — Đóng gói output để tải về

Tải **cả file zip**, không chỉ chép một con số. Toàn bộ manifest, log và score phải về
repo để audit lại được.

In [ ]:
import os, shutil

archive = shutil.make_archive('/kaggle/working/causal_forest_output', 'zip', OUTPUT_ROOT)
print(f'{archive}  {os.path.getsize(archive) / 2**20:.1f} MB')
print()
for root, _, files in os.walk(OUTPUT_ROOT):
    for name in sorted(files):
        p = os.path.join(root, name)
        print(f'{os.path.getsize(p) / 2**20:9.2f} MB  {p}')

## Bước tiếp theo — chạy ở local sau khi tải về

Giải nén vào `output/causal_forest/` rồi chấm điểm:

```powershell
.venv\Scripts\python.exe scripts\evaluate_causal_forest.py `
  --stage-dir output\causal_forest\preflight_0p5 --n-boot 500 --signal dr
```

Với stage 20% và 30%, đổi `--stage-dir` thành `preflight_0p2` / `preflight_0p3`. Script
tự phát hiện và sẽ in `[mode] standalone` kèm cảnh báo rằng holdout đó **không** so được
với bảng release.

## Ba điều không được viết vào báo cáo

1. So Qini stage 20% với `0,187886` của Response — hai tập test khác nhau.
2. "Gate pass nghĩa là model tốt" — gate chỉ kiểm tài nguyên và toàn vẹn artifact;
   `"quality_not_assessed": true` nằm ngay trong manifest.
3. "Causal Forest cho khoảng tin cậy cá nhân" — profile `kaggle-safe` đặt
   `inference=False`, không gọi `effect_interval()` được.